# CGCNN Example: Split `main.py` Training Workflow into Notebook Blocks

This notebook follows the same training logic as `main.py`, but split into clear steps:
1. Load and override config for a fast example run
2. Load data and build dataloaders
3. Build CGCNN model (with checkpoint pretraining initialization)
4. Train for 2 epochs
5. Run prediction and print sample outputs


In [ ]:
from pathlib import Path
import json
import random

import numpy as np
import pandas as pd
import pytorch_lightning as pl
import torch
from pytorch_lightning.callbacks import ModelCheckpoint
from pytorch_lightning.loggers import CSVLogger
from torch.utils.data import DataLoader

from config import get_cfg_defaults
from realmat_bag.loaddata.cifdata import CIFData
from realmat_bag.loaddata.collate import collate_crystal_batch
from realmat_bag.pipeline.models.cgcnn.model_cgcnn import get_cgcnn_model
from realmat_bag.utils.cif_downloader import auto_download_missing_cifs_from_frame

repo_root = Path('.').resolve()
base_cfg = repo_root / 'configs/finetune/cgcnn/cgcnn_10.yaml'
train_out = repo_root / 'saved_models/example/cgcnn_10_pretrain_ckpt_2epochs'
pred_out_dir = repo_root / 'output_test/example_ckpt_pred'
pred_out_file = pred_out_dir / 'predictions.csv'

train_out.mkdir(parents=True, exist_ok=True)
pred_out_dir.mkdir(parents=True, exist_ok=True)

print('Repo root:', repo_root)
print('Base config:', base_cfg)


## Step 1. Load Config and Set Example Hyperparameters

We choose **CGCNN** and reduce training to **2 epochs** so the notebook runs quickly as an example.


In [ ]:
cfg = get_cfg_defaults()
cfg.merge_from_file(str(base_cfg))

cfg.defrost()
cfg.MODEL.NAME = 'cgcnn'
cfg.DATASET.TRAIN = 'data/fine_tune/train_data_10.json'
cfg.DATASET.VAL = 'data/fine_tune/test_data.json'
cfg.MODEL.PRETRAINED_MODEL_PATH = 'saved_models/cgcnn/best.ckpt'
cfg.SOLVER.EPOCHS = 2
cfg.SOLVER.WORKERS = 0
cfg.LOGGING.LOG_DIR = 'saved_models'
cfg.LOGGING.LOG_DIR_NAME = 'example/cgcnn_10_pretrain_ckpt_2epochs'
cfg.freeze()

print('[Example Config]')
print('MODEL.NAME:', cfg.MODEL.NAME)
print('DATASET.TRAIN:', cfg.DATASET.TRAIN)
print('DATASET.VAL:', cfg.DATASET.VAL)
print('SOLVER.EPOCHS:', cfg.SOLVER.EPOCHS)
print('MODEL.PRETRAINED_MODEL_PATH:', cfg.MODEL.PRETRAINED_MODEL_PATH)


## Step 2. Load Data and Build Dataloaders

This mirrors `main.py`: read JSON, ensure CIF files are available, then build `CIFData` + `DataLoader`.


In [ ]:
def load_json_as_dataframe(path):
    with open(path, 'r') as f:
        data = json.load(f)
    df = pd.DataFrame.from_dict(data, orient='index')
    df.reset_index(inplace=True)
    df.rename(columns={'index': 'mpids'}, inplace=True)
    return df

train_df = load_json_as_dataframe(cfg.DATASET.TRAIN)
val_df = load_json_as_dataframe(cfg.DATASET.VAL)

auto_download_missing_cifs_from_frame(
    frame=pd.concat([train_df, val_df], ignore_index=True),
    out_dir=cfg.MODEL.CIF_FOLDER,
    auto_download=True,
)

train_dataset = CIFData(
    train_df[['mpids', 'bg']],
    cfg.MODEL.CIF_FOLDER,
    cfg.MODEL.INIT_FILE,
    cfg.MODEL.MAX_NBRS,
    cfg.MODEL.RADIUS,
    cfg.SOLVER.RANDOMIZE,
)
val_dataset = CIFData(
    val_df[['mpids', 'bg']],
    cfg.MODEL.CIF_FOLDER,
    cfg.MODEL.INIT_FILE,
    cfg.MODEL.MAX_NBRS,
    cfg.MODEL.RADIUS,
    cfg.SOLVER.RANDOMIZE,
)

train_loader = DataLoader(
    train_dataset,
    collate_fn=collate_crystal_batch,
    batch_size=cfg.SOLVER.BATCH_SIZE,
    shuffle=True,
    num_workers=cfg.SOLVER.WORKERS,
)
val_loader = DataLoader(
    val_dataset,
    collate_fn=collate_crystal_batch,
    batch_size=cfg.SOLVER.BATCH_SIZE,
    shuffle=False,
    num_workers=cfg.SOLVER.WORKERS,
)

print('Train samples:', len(train_dataset))
print('Val samples:', len(val_dataset))


## Step 3. Build CGCNN Model and Load Pretrained Checkpoint

We infer feature dimensions from one sample (same as `main.py`) and initialize from a checkpoint.


In [ ]:
def set_random_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_random_seed(cfg.SOLVER.SEED)

sample = train_dataset[0]
cfg.defrost()
cfg.CGCNN.ORIG_ATOM_FEA_LEN = sample.atom_fea.shape[-1]
cfg.CGCNN.NBR_FEA_LEN = sample.nbr_fea.shape[-1]
cfg.CGCNN.POS_FEA_LEN = sample.positions.shape[-1]
cfg.freeze()

model = get_cgcnn_model(cfg)

ckpt_path = Path(cfg.MODEL.PRETRAINED_MODEL_PATH)
if ckpt_path.exists():
    checkpoint = torch.load(str(ckpt_path), map_location='cpu')
    state_dict = checkpoint['state_dict'] if 'state_dict' in checkpoint else checkpoint
    model.load_state_dict(state_dict, strict=False)
    print('Loaded pretrained checkpoint:', ckpt_path)
else:
    print('No pretrained checkpoint found; training from scratch.')

print('Model class:', model.__class__.__name__)
print('Backbone class:', model.model.__class__.__name__)


## Step 4. Train for 2 Epochs

This is the training block equivalent to `main.py` for one example run.


In [ ]:
best_mre_checkpoint = ModelCheckpoint(
    dirpath=str(train_out),
    monitor='val_mre',
    mode='min',
    save_top_k=1,
    filename='example-best-mre-{epoch:02d}-{val_mre:.4f}',
)
last_checkpoint = ModelCheckpoint(
    dirpath=str(train_out),
    save_last=True,
    filename='example-last-{epoch:02d}',
)

csv_logger = CSVLogger(save_dir='logs', name='example_cgcnn_2ep')
trainer = pl.Trainer(
    max_epochs=cfg.SOLVER.EPOCHS,
    accelerator='cpu',
    devices=1,
    logger=csv_logger,
    callbacks=[best_mre_checkpoint, last_checkpoint],
)

trainer.fit(model, train_dataloaders=train_loader, val_dataloaders=val_loader)

best_ckpt = Path(best_mre_checkpoint.best_model_path) if best_mre_checkpoint.best_model_path else None
print('Best checkpoint:', best_ckpt)


## Step 5. Predict and Print Prediction Examples

We run inference on the validation/test loader, print a few rows, and save CSV.


In [ ]:
if best_ckpt is not None and best_ckpt.exists():
    best_state = torch.load(str(best_ckpt), map_location='cpu')['state_dict']
    model.load_state_dict(best_state, strict=False)

model.eval()
rows = []
with torch.no_grad():
    for batch in val_loader:
        outputs = model(batch)
        if isinstance(outputs, (tuple, list)):
            outputs = outputs[0]
        preds = outputs.detach().cpu().view(-1).numpy()
        gts = batch.target.detach().cpu().view(-1).numpy()
        mpids = list(batch.cif_ids)
        for mpid, gt, pred in zip(mpids, gts, preds):
            rows.append({'mpids': mpid, 'bg': float(gt), 'prediction': float(pred)})

pred_df = pd.DataFrame(rows)
pred_df.to_csv(pred_out_file, index=False)

print('Prediction file:', pred_out_file)
print('Total rows:', len(pred_df))
print(pred_df.head(10).to_string(index=False))
